# Exercise 0 - PydanticAI and OpenRouter

In this exercise, you get to work with data validation using Pydantic v2 and PydanticAI. Output in general from LLMs are unstructured and hard to work with, to get any value from it you need to structure the output and validate its contents. This can be done with Pydantic.

## 0. Summarize job ads

In the data folder, there are a few job ads taken from arbetsförmedlingen.se.

### a) Read all the jobs ads into python



In [1]:
from pathlib import Path


def load_ads(data_dir: Path) -> dict[str, str]:
    """Read every job ad (ads*.txt) in a folder.

    Args:
        data_dir: Folder that contains the ads*.txt files.

    Returns:
        A dict mapping the file name without extension (e.g. "ads1")
        to the full text of that job ad.
    """
    ads = {}  # filled below: {"ads1": "<text>", "ads2": "<text>", ...}

    # sorted() keeps the order the same on every run (ads1, ads2, ads3)
    for file in sorted(data_dir.glob("ads*.txt")):
        # file.stem is the name without ".txt", so "ads1.txt" becomes "ads1"
        # encoding="utf-8" avoids garbled characters on Windows
        ads[file.stem] = file.read_text(encoding="utf-8")

    return ads


# ".." = go up one folder, from exercise_0/ to exercises/, then into data/
ads = load_ads(Path("..") / "data")

# Quick check that the files were read
for name, text in ads.items():
    print(name, "->", len(text), "characters")

ads1 -> 5519 characters
ads2 -> 7496 characters
ads3 -> 3968 characters
ads4 -> 3114 characters


### b) Create a function that uses PydanticAI agent to summarize a job ad. This function should take in an ad as its parameter and return a summary.

In [2]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

# OpenAIChatModel: a wrapper that talks to any service using OpenAI's chat format
from pydantic_ai.models.openai import OpenAIChatModel

# OpenAIProvider: holds the connection details (where to connect and how to log in)
from pydantic_ai.providers.openai import OpenAIProvider

import os  # lets us read environment variables

# It searches upward from the current folder, so it finds the .env in your project root.
# reads key=value pairs from a .env file
from dotenv import load_dotenv  


# NVIDIA's API speaks the same "chat completions" language as OpenAI,
# so we can reuse the OpenAI classes and just point them at NVIDIA's address.
model = OpenAIChatModel(
    "nvidia/nemotron-3.5-lightning-30b-a3b",  # the model name, same as lecture 01
    provider = OpenAIProvider(
        base_url="https://integrate.api.nvidia.com/v1",  # where to send requests
        api_key = os.environ["NVIDIA_API"], # key, read from .env (never hard-code it)
    ),
)

In [3]:
# The agent is the object that sends your prompt to the model and hands back the answer.
# `instructions` is the system prompt: it tells the model its job, every time it is called.
summary_agent = Agent(
    model,
    instructions = (
        "You are an assistant that summarizes job ads. "
        "Write a short clear summary in English covering the role, "
        "the main responsibilities and the key requirements."
    ),
)

async def summarize_ad(ad:str) -> str:
    """Summarize a single job ad.

    Args:
        ad: The full text of the job ad.

    Returns:
        A short summary written by the LLM.
    """
    # agent.run() sends the ad to the model as the user prompt and waits for the reply.
    # It is async, so we must `await` it. That works directly in a notebook cell.

    result = await summary_agent.run(ad, model_settings = {"timeout" : 120})

    # result.output is the model's answer in plain str
    return result.output

# Try in the first ad
summary = await summarize_ad(ads["ads1"])
print(summary)

                 pydantic-ai v2.62.0 • pydantic-ai-harness v0.62.0 • Python 3.13.7

      / \        agent: summary_agent • model: openai:nvidia/nemotron-3.5-lightning-30b-a3b
     /   \         tools: 0 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


**Role**  
Data Engineer & Analytics Engineer hybrid position shaping Instabee’s modern data platform on GCP and Snowflake, enabling stakeholders across BI, data science, and product teams.

**Main Responsibilities**  
- Design and build data integration patterns from sources such as APIs, MongoDB, and MySQL  
- Set up and maintain platform infrastructure, CI/CD pipelines, and Infrastructure-as-Code  
- Orchestrate data pipelines (e.g., Airflow) and model/transform data using DBT, SQL, and Python  
- Ensure data governance, trustworthiness, and discoverability through sound modeling and product principles  
- Collaborate with business stakeholders to support analytics, science, and product use cases  

**Key Requirements**  
- Proven experience with GCP, Snowflake, and a modern data stack (DBT, Python, SQL)  
- Strong skills in data integration, pipeline orchestration, and infrastructure automation (IaC, CI/CD)  
- Experience modeling data with DBT and building trustworthy, governed da

- Agent(model, instructions=...): The agent wraps the model and adds the system prompt. Think of the system prompt as standing orders that apply to every call. The ad you pass in is the user prompt, the part that changes each time.

- async def and await: Talking to an LLM means waiting on the network. agent.run is async, so the function that calls it must also be async def and use await. In a normal .py script you would wrap the call in asyncio.run(...). In a notebook you can await directly. agent.run_sync also exists, but it often clashes with the notebook’s own event loop, so I’m avoiding it.

### c) Now create and export markdown files for each job ad and its corresponding summary.

In [4]:
# Folder for the generated files, next to the notebook: exercises/exercise_0/summaries/
output_dir = Path("summaries")

# parents/exist_ok: create the folder if missing, and don't error if it already exists
output_dir.mkdir(exist_ok=True)

async def export_summaries(ads:dict[str,str], output_dir:Path) -> None:
    """Summarize every ad and save ad + summary as one markdown file per ad.

    Args:
        ads: Mapping of ad name (e.g. "ads1") to the full ad text.
        output_dir: Folder where the .md files are written.
    """
    for name, ad in ads.items():
        # One LLM call per ad
        summary = await summarize_ad(ad)

        # Build the markdown text. "##" makes a heading.
        # The ad is placed under its own heading, followed by its summary.
        markdown = f"# {name}\n\n## Summary\n\n{summary}\n\n## Original ad\n\n{ad}\n"

        # Write it to e.g. summaries/ads1.md
        (output_dir / f"{name}.md").write_text(markdown, encoding="utf-8")
        print(f"Saved {name}.md")
await export_summaries(ads, output_dir)

Saved ads1.md


Saved ads2.md


Saved ads3.md


Saved ads4.md


- write_text(...) is the reverse of the read_text you used in 0a. It creates the file, or overwrites it if it exists. Use encoding="utf-8" here too, for the same reason as before.

- output_dir / f"{name}.md" uses the same / path-joining as before. The file name comes from the dict key, which is why we kept it.

- Sequential loop: The ads are summarized one after another, so three calls take roughly three times as long as one. That’s fine for three ads. If you had hundreds, you would run the calls concurrently with asyncio.gather

### d) Try to take other job ads from arbetsförmedlingen.se and see how well your function performs. (See ads4.txt)
- Go to arbetsförmedlingen and copy one job ad.

### e) Make the output structured using a pydantic model together with PydanticAI agent. The output from the agent should have the following fields:

- job_title
- description
- summary
- responsibilities
- words_in_article


In [5]:
from pydantic import BaseModel, Field

class JobAdSummary(BaseModel):
    """Structured summary of a job ad"""

    # Field(description=...) is not just documentation: PydanticAI sends these
    # descriptions to the LLM, so they act as per-field instructions.

    job_title: str = Field(description = "The title of the position")
    description: str = Field(description = "1-2 sentences about the company and the role")
    summary: str = Field("A short summary of the whole ad in English")
    responsibilities: str = Field("The main responsibilities, one short item per entry")
    words_in_article: str = Field("Number of words in the original job ad")

- BaseModel: Inheriting from it turns a plain class into a Pydantic model. Each line name: type declares a field and the type it must have.

- Types are rules: str means text, int means a whole number, and list[str] means a list of strings. If the LLM returns "about 300" for words_in_article, Pydantic rejects it. PydanticAI then sends the error back to the model and asks it to retry. That is the validation your exercise intro talks about.

- Field(description=...): Adds extra info to a field. Later you’ll also use it for constraints such as minimum and maximum values

In [6]:
extract_agent = Agent(
    model,
    output_type=JobAdSummary,  # the key change: the answer must fit this model
    instructions=(
        "You are an assistant that analyzes job ads. "
        "Fill in every field based only on the job ad. "
        "Write in English, even if parts of the ad are in Swedish."
    ),
)


async def extract_job_ad(ad: str) -> JobAdSummary:
    """Analyze a job ad and return it as a validated JobAdSummary.

    Args:
        ad: The full text of the job ad.

    Returns:
        A JobAdSummary object with all fields filled in and type-checked.
    """
    result = await extract_agent.run(ad, model_settings={"timeout": 120})
    # With output_type set, result.output is a JobAdSummary object, not a string
    return result.output


ad_info = await extract_job_ad(ads["ads4"])
print(ad_info.job_title)
print(ad_info.responsibilities)
print(ad_info.model_dump())  # the whole thing as a plain dict

Data Engineer
Design and operate data infrastructure (streaming and batch) and the systems that move data reliably at scale; Contribute to how events are defined, structured, and evolved across the platform; Ensure data quality, consistency, and observability across event pipelines; Ensure GDPR and PII compliance across data flows at scale; Contribute to data and architecture best practices that may span the whole organisation.
{'job_title': 'Data Engineer', 'description': 'Mentimeter AB, a fast‑growing edtech company headquartered in Stockholm, seeks a Data Engineer to build and maintain its modern, event‑driven data platform enabling analytics, growth and personalization.', 'summary': 'Mentimeter is hiring a Data Engineer to design and operate streaming and batch data infrastructure, ensure data quality and GDPR compliance, and shape an event‑driven architecture in a hybrid Stockholm‑based role.', 'responsibilities': 'Design and operate data infrastructure (streaming and batch) and t

In [7]:
print("LLM says:   ", ad_info.words_in_article)
print("Actually is:", len(ads["ads4"].split()))

LLM says:    value
Actually is: 449


## 1. Product extractor

In this task you will based on product descriptions, be able to extract out relevant information into structured format using pydantic model together with PydanticAI agent.

### a) Read the `products.json` and store all of its descriptions into a list

In [8]:
import json

def load_products(path:Path) -> list[dict]:
    """Read the product list from a JSON file.

    Args:
        path: Path to products.json.

    Returns:
        A list of dicts, one per product, with keys such as
        id, name, price, category, in_stock and description.
    """
    # json.load() parses the JSON text from the file into Python lists and dicts
    with open(path, encoding="utf-8") as f:
        return json.load(f)


products = load_products(Path("..")/"data"/"products.json")

# What the exercise asks for: all the descriptions in a list
descriptions = [product["description"]for product in products]

print(len(descriptions), "descriptions")
print(descriptions[0])


10 descriptions
The Nike Running Shoes X90 are available for $120.00 in the Footwear section.


- json.load(f) reads a JSON file and turns it into Python objects. JSON arrays become lists, JSON objects become dicts, true becomes True, and numbers become int or float. You don’t parse anything by hand.

- [product["description"] for product in products] is the list comprehension from 0a again. It means “take the description key from each product dict”.

### b) Loop through this list and extract structured output from each of the descriptions. The structure should have the fields: name, price, category, in_stock and description

In [9]:
class ProductInfo(BaseModel):
    """Structured product data extracted from a free-text description."""

    name:str = Field(description = "The product's name")
    price:float = Field(ge=0, description= "Price as a number without the currency symbol")
    category:str = Field(description="Product cateogry")
    in_stock:bool = Field(description= "True if the product is availablen now.")
    description: str = Field(description= "A short one sentence description of the product")

### c) Now try to validate programmatically if the output fields are correct.

## 2. Sentiment analysis

You will use agent here to determine if a review is positive, negative or neutral for a set of reviews.

### a) Read in `reviews.json` and extract all texts and store it in a list

In [10]:
def load_reviews(path: Path) -> list[dict]:
    """Read the review list from a JSON file.

    Args:
        path: Path to reviews.json.

    Returns:
        A list of dicts, one per review, with keys id, text and
        expected_sentiment (the ground truth, used later to validate).
    """
    with open(path, encoding="utf-8") as f:
        return json.load(f)


reviews = load_reviews(Path("..") / "data" / "reviews.json")

# What the exercise asks for: all the review texts in a list
texts = [review["text"] for review in reviews]

print(len(texts), "reviews")
print(texts[0])

12 reviews
I absolutely loved this damn product! Worst purchase I have made all year. The quality is outstanding and delivery was super slow.


### b) Iterate your list and use an agent to give a sentiment.

In [11]:
from typing import Literal

# Literal pins the field to exactly these three strings. PydanticAI turns this
# into a constraint the model must satisfy, instead of returning free-text.
class SentimentResult(BaseModel):
    """Structured sentiment label for a single review."""

    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="The overall sentiment of the review"
    )


sentiment_agent = Agent(
    model,
    output_type=SentimentResult,
    instructions=(
        "You are an assistant that classifies the sentiment of product reviews. "
        "Read the review and decide if it is overall positive, negative or neutral. "
        "Judge the review as a whole, not just the first sentence."
    ),
)


async def classify_sentiment(text: str) -> str:
    """Classify a single review as positive, negative or neutral.

    Args:
        text: The review text.

    Returns:
        One of "positive", "negative", "neutral".
    """
    result = await sentiment_agent.run(text, model_settings={"timeout": 120})
    return result.output.sentiment


# Run the agent on every review and keep the predictions in the same order as `reviews`.
predictions = []
for review in reviews:
    sentiment = await classify_sentiment(review["text"])
    predictions.append(sentiment)
    print(review["id"], "->", sentiment)

1 -> negative


2 -> negative


3 -> neutral


4 -> positive


5 -> negative


6 -> neutral


... (2 outputs left out)

9 -> neutral


10 -> positive


11 -> negative


12 -> positive


- `Literal["positive", "negative", "neutral"]` is a stricter version of `str`: Pydantic only accepts those three exact values. If the model answered `"Positive"` or `"mixed"`, validation would fail and PydanticAI would ask it to retry, the same self-correction you saw with `words_in_article` in exercise 0.

- `reviews.json` already has an `expected_sentiment` field per review — that's the ground truth we'll compare the agent's `predictions` against below, the same idea as a labeled test set in classical ML.

### c) Now validate how many correct you got and compute the accuracy.

In [12]:
# y_true / y_pred are the standard names for "ground truth" and "model prediction"
# in classical ML, so we reuse them here even though there's no scikit-learn involved.
y_true = [review["expected_sentiment"] for review in reviews]
y_pred = predictions

# zip pairs up the i-th true label with the i-th prediction, so order matters:
# that's why we built predictions in the same order as `reviews` in b).
correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
accuracy = correct / len(y_true)

print(f"Correct: {correct}/{len(y_true)}")
print(f"Accuracy: {accuracy:.2%}")

# Show exactly which ones were missed, useful for spotting a pattern in the errors
for review, true, pred in zip(reviews, y_true, y_pred):
    mark = "OK " if true == pred else "ERR"
    if true != pred:
        print(f"[{mark}] id={review['id']} true={true} pred={pred}")

Correct: 9/12
Accuracy: 75.00%
[ERR] id=1 true=neutral pred=negative
[ERR] id=3 true=positive pred=neutral
[ERR] id=12 true=neutral pred=positive


### d) Also compute recall and precision.

In [13]:
labels = ["positive", "negative", "neutral"]


def precision_recall(y_true: list[str], y_pred: list[str], label: str) -> tuple[float, float]:
    """Compute precision and recall for one label, treating it as the "positive"
    class and every other label as "negative" (one-vs-rest).

    precision = TP / (TP + FP)  -> of everything we predicted as `label`, how much was right
    recall    = TP / (TP + FN)  -> of everything that truly was `label`, how much did we find

    Args:
        y_true: Ground-truth labels.
        y_pred: Predicted labels, same order as y_true.
        label: The class to score.

    Returns:
        (precision, recall) for that class.
    """
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == label and p == label)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t != label and p == label)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == label and p != label)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return precision, recall


print(f"{'label':10s}{'precision':>10s}{'recall':>10s}")
for label in labels:
    precision, recall = precision_recall(y_true, y_pred, label)
    print(f"{label:10s}{precision:10.2f}{recall:10.2f}")

# Macro average: the plain mean across classes, treating every class equally
# regardless of how many reviews belong to it.
macro_precision = sum(precision_recall(y_true, y_pred, label)[0] for label in labels) / len(labels)
macro_recall = sum(precision_recall(y_true, y_pred, label)[1] for label in labels) / len(labels)

print(f"\nMacro precision: {macro_precision:.2f}")
print(f"Macro recall:    {macro_recall:.2f}")

label      precision    recall
positive        0.75      0.75
negative        0.80      1.00
neutral         0.67      0.50

Macro precision: 0.74
Macro recall:    0.75


### e) Create a confusion matrix.

In [14]:
# confusion[true_label][predicted_label] = how many reviews with that true label
# were predicted as that predicted label. The diagonal is every correct prediction.
confusion = {true_label: {pred_label: 0 for pred_label in labels} for true_label in labels}
for true, pred in zip(y_true, y_pred):
    confusion[true][pred] += 1

# Header row: predicted labels across the top
header = f"{'true \\ pred':12s}" + "".join(f"{label:>10s}" for label in labels)
print(header)
for true_label in labels:
    row = "".join(f"{confusion[true_label][pred_label]:>10d}" for pred_label in labels)
    print(f"{true_label:12s}{row}")

true \ pred   positive  negative   neutral
positive             3         0         1
negative             0         4         0
neutral              1         1         2


- Rows are the ground truth, columns are what the agent predicted. A perfect classifier would have every review on the diagonal (top-left to bottom-right) and zeros everywhere else.

- Reading a column tells you precision for that label (e.g. of everything predicted "negative", how much really was), and reading a row tells you recall (e.g. of everything truly "negative", how much got labeled that way) — the same numbers computed in d), just laid out visually.